# DeepPEF: Protein Stability Prediction (ddG)

**M.Sc. Thesis - Nissim Brami**

Full training pipeline: HuggingFace data → GNN training → 5-seed ensemble.

**Proven config:** PCC = 0.5259 (single seed), expected 0.55-0.57 ensemble.

**Runtime:** ~11h per seed on T4, ~6h on A100. Ensemble = 5 seeds.

**Critical:** `mini_batch_size=64` is REQUIRED for correct ranking loss signal.

## 0. GPU Check

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU! Runtime > Change runtime type > GPU")

## 1. Clone Repo & Install Dependencies

In [ ]:
import os

REPO = "https://github.com/nissimbrami/DeepEF-Thesis.git"
WORK = "/content/DeepPEF"

if not os.path.exists(WORK):
    !git clone {REPO} {WORK}
os.chdir(WORK)
print(f"Working dir: {os.getcwd()}")

In [ ]:
!pip install -q torch-geometric transformers fair-esm wandb scipy scikit-learn tqdm pandas huggingface_hub
import torch
tv = torch.__version__.split("+")[0]
cv = torch.version.cuda.replace(".", "") if torch.version.cuda else "cpu"
!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{tv}+cu{cv}.html
os.environ["WANDB_MODE"] = "disabled"
print("All dependencies installed.")

## 2. Download Data from HuggingFace

Data sources:
- `nissimb/deepef-megascale` — 368 MegaScale proteins with per-mutation ProtT5 embeddings
- `shaharec/deepef-data` — CASP12 pretraining data (16K proteins, optional)

The MegaScale data is all we need for fine-tuning (PCC 0.5259).

In [ ]:
from huggingface_hub import snapshot_download
import shutil

# Download MegaScale data from HuggingFace
HF_REPO = "nissimb/deepef-megascale"
LOCAL_DATA = "/content/hf_data"

print("Downloading MegaScale data from HuggingFace...")
print("(This is ~83GB, may take 20-40 min on Colab)")
snapshot_download(
    repo_id=HF_REPO,
    repo_type="dataset",
    local_dir=LOCAL_DATA,
    local_dir_use_symlinks=False
)
print(f"Downloaded to {LOCAL_DATA}")

In [ ]:
# Set up data directory structure
import os

# The HF download should have training_data/ and mutation_files/
# Link them to where the code expects them
os.makedirs("data/MsDs", exist_ok=True)
os.makedirs("data/ThermoMPNN", exist_ok=True)
os.makedirs("data/Processed_K50_dG_datasets/Pnas_filtering", exist_ok=True)

# Find the actual data location in HF download
hf_base = LOCAL_DATA

# Look for training_data directory
for root, dirs, files in os.walk(hf_base):
    if "training_data" in dirs:
        td_path = os.path.join(root, "training_data")
        break
else:
    # Try direct path
    td_path = os.path.join(hf_base, "training_data")

for root, dirs, files in os.walk(hf_base):
    if "mutation_files" in dirs:
        mf_path = os.path.join(root, "mutation_files")
        break
else:
    mf_path = os.path.join(hf_base, "mutation_files")

# Create symlinks
for link, target in [
    ("data/MsDs/training_data", td_path),
    ("data/MsDs/mutation_files", mf_path),
]:
    if os.path.exists(link):
        os.remove(link) if os.path.islink(link) else shutil.rmtree(link)
    os.symlink(target, link)
    print(f"  {link} -> {target}")

# Find and link supporting files
for root, dirs, files in os.walk(hf_base):
    for f in files:
        if f == "mega_test.csv":
            src = os.path.join(root, f)
            dst = "data/ThermoMPNN/mega_test.csv"
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                print(f"  Copied {f}")
        elif f == "train_proteins.csv":
            src = os.path.join(root, f)
            dst = "data/Processed_K50_dG_datasets/Pnas_filtering/train_proteins.csv"
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                print(f"  Copied {f}")
        elif f == "pnas_mutations.csv":
            src = os.path.join(root, f)
            dst = "data/Processed_K50_dG_datasets/Pnas_filtering/pnas_mutations.csv"
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                print(f"  Copied {f}")

print("\nData setup complete.")

In [ ]:
# Verify data
import torch, os
td = "./data/MsDs/training_data"
mf = "./data/MsDs/mutation_files"

proteins = sorted([d for d in os.listdir(td) if os.path.isdir(os.path.join(td, d))])
mut_files = [f for f in os.listdir(mf) if f.endswith(".csv")]
print(f"Proteins: {len(proteins)}")
print(f"Mutation files: {len(mut_files)}")

# Check first protein has all required files
p = proteins[0]
required = ["coords_tensor.pt", "deltaG.pt", "mask_tensor.pt", "one_hot_encodings.pt"]
emb_dir = os.path.join(td, p, "prott5_embeddings")
print(f"\nFirst protein: {p}")
for f in required:
    exists = os.path.exists(os.path.join(td, p, f))
    print(f"  {f}: {'OK' if exists else 'MISSING'}")
if os.path.exists(emb_dir):
    n_embs = len([f for f in os.listdir(emb_dir) if f.endswith(".pt")])
    print(f"  prott5_embeddings/: {n_embs} files")

# Check ThermoMPNN test file
tm_path = "./data/ThermoMPNN/mega_test.csv"
pnas_path = "./data/Processed_K50_dG_datasets/Pnas_filtering/train_proteins.csv"
print(f"\nmega_test.csv: {'OK' if os.path.exists(tm_path) else 'MISSING'}")
print(f"train_proteins.csv: {'OK' if os.path.exists(pnas_path) else 'MISSING'}")

if torch.cuda.is_available():
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 3. Remove Cache Files (Prevent Bugs)

Stale cache files cause silent data loading errors.

In [ ]:
import glob, shutil

# Remove any .cache files that cause bugs
cache_csv = "./data/MsDs/mutation_files/.cache.csv"
cache_dir = "./data/MsDs/training_data/.cache"

if os.path.exists(cache_csv):
    os.remove(cache_csv)
    print(f"Removed {cache_csv}")
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"Removed {cache_dir}")

# Clear any __pycache__
for p in glob.glob("./**/__pycache__", recursive=True):
    shutil.rmtree(p)
print("Cache cleared.")

## 4. Train: 5-Seed Ensemble (Proven PCC=0.5259 Config)

This is the EXACT config that achieved PCC=0.5259 on May 28:
- k-NN GAT (k=30, dynamic cutoff)
- Huber + Ranking loss (weight=0.1)
- Cosine LR (1e-4 → 1e-6)
- Weight decay 1e-5
- **mini_batch_size=64** (CRITICAL — 16 drops PCC by 0.07)
- PNAS filtering (--one_mut --dg_ml)
- From scratch (--no_pretrained)
- 15 epochs per seed

In [ ]:
%%time
import os
os.makedirs("logs", exist_ok=True)
os.makedirs("Megascale-fineTuning/models", exist_ok=True)

seeds = [42, 123, 456, 789, 1337]

for seed in seeds:
    print(f"\n{'='*60}")
    print(f"  TRAINING SEED {seed}")
    print(f"{'='*60}")
    !python Megascale-fineTuning/pnas_train.py \
        --model_name baseline_seed{seed} \
        --seed {seed} \
        --dataset_type pnas \
        --epochs 15 \
        --epochs_freeze 15 \
        --epochs_unfreeze 0 \
        --no_pretrained \
        --loss_type huber_rank \
        --ranking_weight 0.1 \
        --use_knn_gat \
        --one_mut \
        --dg_ml \
        --cosine_lr \
        --lr_min 1e-6 \
        --weight_decay 1e-5 \
        --mini_batch_size 64 \
        2>&1 | tee logs/baseline_seed{seed}.log
    
    # Print result
    !grep -i "Best Pearson\|Training completed" logs/baseline_seed{seed}.log | tail -1

## 5. Results Summary

In [ ]:
import re

print("="*60)
print("  RESULTS: 5-Seed Ensemble")
print("="*60)

pccs = []
for seed in seeds:
    log_path = f"logs/baseline_seed{seed}.log"
    best_pcc = 0.0
    if os.path.exists(log_path):
        with open(log_path) as f:
            for line in f:
                # Match PCC values
                matches = re.findall(r'PCC[^\d]*(\d+\.\d+)', line)
                for m in matches:
                    val = float(m)
                    if 0.1 < val < 1.0 and val > best_pcc:
                        best_pcc = val
                # Also match "Pearson Correlation: X.XXX"
                matches = re.findall(r'Pearson Correlation: (\d+\.\d+)', line)
                for m in matches:
                    val = float(m)
                    if val > best_pcc:
                        best_pcc = val
    pccs.append(best_pcc)
    print(f"  Seed {seed:5d}: PCC = {best_pcc:.4f}")

if pccs and max(pccs) > 0:
    print(f"\n  Average PCC: {sum(pccs)/len(pccs):.4f}")
    print(f"  Best single: {max(pccs):.4f}")
    print(f"  Expected ensemble: ~{sum(pccs)/len(pccs) + 0.02:.4f}")
print("="*60)

## 6. Save Results to Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

SAVE = "/content/drive/MyDrive/DeepPEF/results"
os.makedirs(SAVE, exist_ok=True)

# Save logs
!cp -r logs/ {SAVE}/
# Save model checkpoints
!cp -r Megascale-fineTuning/models/ {SAVE}/

print(f"Results saved to {SAVE}")
!ls {SAVE}/logs/

## Summary

| Metric | Value |
|--------|-------|
| Config | k-NN GAT + Huber+Rank + cosine LR + WD |
| mini_batch_size | **64** (critical) |
| Single seed target | PCC ~0.52 |
| 5-seed ensemble target | PCC ~0.55-0.57 |
| Proven result | PCC = 0.5259 (seed 42, May 28) |

### What makes this work:
1. **k-NN GAT** — biggest single gain (+0.062)
2. **mini_batch_size=64** — ranking loss needs 2016 pairs (not 120 with batch=16)
3. **Huber loss** — handles outlier ddG measurements
4. **PNAS filtering** — quality > quantity for noisy labels
5. **From scratch** — pretrained decoy weights hurt ddG prediction